# Fraud Detection on Card Transaction Data (Under Sampling)

Machine learning solution for Credit Card Fraud Detection.

The pipeline will include `Decision Tree` and `Random Forest` models, along with `NearMiss` (under-sampling) 
technique on the **negative** (i.e., non-fraud samples) to deal with imbalanced data points.

### Loading Data

In [ ]:
import pandas as pd
import os

project_dir = os.getenv("PROJECT_DIR")
env = os.getenv("CONDA_DEFAULT_ENV")
dataset_csv = os.getenv("DATA")

# Ignore User Warnings
os.environ["PYTHONWARNINGS"] = "ignore"
import warnings

warnings.filterwarnings("ignore")

In [ ]:
print(dataset_csv)

In [ ]:
df = pd.read_csv(dataset_csv)

In [ ]:
df.head()

In [ ]:
df.Class.value_counts()

In [ ]:
X, y = df[df.columns[df.columns != "Class"]], df["Class"]

In [ ]:
X.shape, y.shape

## Experimental Pipeline

In [ ]:
# Reproducibility settings
import numpy as np
from sklearn.utils import check_random_state

SEED = 12345

# The NumPy Generator will be used throughout the whole experiment
# rng = np.random.default_rng(SEED)
np.random.seed(SEED)
rng = check_random_state(SEED)

In [ ]:
# Preprocessing
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer

# Imbalanced Learning
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import NearMiss

# Model Selection and Metrics
from sklearn.model_selection import train_test_split

# ML Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

**Data Splitting**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=rng)

**PreProcessing**

In [ ]:
# (Selected) Feature Scaling
preprocessing = ColumnTransformer(
    [
        ("scaler", RobustScaler(), ["Time", "Amount"]),
    ],
    remainder="passthrough",
)

### Machine Learning Models

Setting up Machine Learning models and their corresponding param grid (for Hyper parameter tuning)

In [ ]:
# Decision Tree
dt = DecisionTreeClassifier(random_state=rng)
tree_models_params = {
    "model__max_depth": [None, 2, 3, 6],
    "model__min_samples_leaf": [2, 5, 6],
    "model__criterion": ["gini", "entropy"],
}

dt_params = tree_models_params

In [ ]:
# Random Forest
rf = RandomForestClassifier(random_state=rng, n_jobs=-1)
rf_params = {
    "model__n_estimators": [
        50,
    ],
    "model__max_features": ["log2", "sqrt"],
}
rf_params_full = tree_models_params | rf_params

#### Near-Miss (Under) Sampling Strategy

In [ ]:
nm_run_config = [
    ("Decision Tree", dt, dt_params),
    ("Random Forest", rf, rf_params_full),
]

In [ ]:
# Under Sampling Strategy
nm = NearMiss(sampling_strategy="majority", version=3)
# NearMiss Param Grid
nm_params = {"sampling__n_neighbors_ver3": [4, 5]}

steps_under_sampling = [("preprocess", preprocessing), ("sampling", nm)]

In [ ]:
from fraud_detection.notebook.train import run_experiment

run_experiment(
    name="Under Sampling Near Miss",
    model_configs=nm_run_config,
    data=(X_train, X_test),
    labels=(y_train, y_test),
    preprocessing_steps=steps_under_sampling,
    preproc_hyper_params=nm_params,
    rng=rng,
)

---